### Data Ingestion

In [1]:
### Document data structure
from langchain_core.documents import Document


In [2]:
doc=Document(page_content="This is the content of the document  I am creating for RAG.", 
             metadata={"source": "example.txt",
                       "author": "Niaz",
                       "date": "2024-10-01"
                       }
                       
            )
doc

Document(metadata={'source': 'example.txt', 'author': 'Niaz', 'date': '2024-10-01'}, page_content='This is the content of the document  I am creating for RAG.')

In [3]:
### create simple txt file
import os
os.makedirs("data/text_files", exist_ok=True)

In [18]:
sample_text ={"data/text_files/RAG_intro.txt": """Retrieval-Augmented Generation (RAG) is an advanced technique in natural 
language processing that combines retrieval-based methods with generative models to enhance 
the quality and relevance of generated text. The core idea behind RAG is to leverage a large 
corpus of documents or knowledge base to retrieve relevant information that can inform and guide the generation process.

"""
}

for file_path, content in sample_text.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)




In [5]:
### TextLoader to load txt files
##from langchain.document_loaders import TextLoader

##Alternative way to create a loader for txt files
from langchain_community.document_loaders import TextLoader   

loader = TextLoader("data/text_files/RAG_intro.txt",encoding='utf-8')
documents = loader.load()
documents[0]

Document(metadata={'source': 'data/text_files/RAG_intro.txt'}, page_content='Retrieval-Augmented Generation (RAG) is an advanced technique in natural \nlanguage processing that combines retrieval-based methods with generative models to enhance \nthe quality and relevance of generated text. The core idea behind RAG is to leverage a large \ncorpus of documents or knowledge base to retrieve relevant information that can inform and guide the generation process.\n\n')

In [7]:
### Directory Loader to load multiple files from a directory
from langchain.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
## load all txt files from the directory
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="*.txt", ##pattern to match files
    loader_cls=TextLoader, ##specify the loader class to use
    loader_kwargs={"encoding": "utf-8"}, ##additional arguments for the loader
    show_progress=True ##show progress bar
)
dir_documents = dir_loader.load()
dir_documents[0]

100%|██████████| 1/1 [00:00<?, ?it/s]


Document(metadata={'source': 'data\\text_files\\RAG_intro.txt'}, page_content='Retrieval-Augmented Generation (RAG) is an advanced technique in natural \nlanguage processing that combines retrieval-based methods with generative models to enhance \nthe quality and relevance of generated text. The core idea behind RAG is to leverage a large \ncorpus of documents or knowledge base to retrieve relevant information that can inform and guide the generation process.\n\n')

In [8]:
### Directory Loader to pdf files
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader
# Note: PyMuPDF (pymupdf) must be installed for PyMuPDFLoader to work: pip install pymupdf
pdf_loader = DirectoryLoader(
    "data/pdf",
    glob="*.pdf", ##pattern to match files
    loader_cls=PyMuPDFLoader, ##specify the loader class to use
    show_progress=False##show progress bar
)
pdf_documents = pdf_loader.load()
pdf_documents

[Document(metadata={'producer': 'Skia/PDF m118 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'data\\pdf\\170 Python projects.pdf', 'file_path': 'data\\pdf\\170 Python projects.pdf', 'total_pages': 15, 'format': 'PDF 1.4', 'title': '170 Python Projects', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='170 Python\nProjects\nwith Source Code & Resources\nA collection of simple Python mini projects to enhance your Python skills.\nHimanshu Ramchandani\nhttps://www.linkedin.com/in/hemansnation/'),
 Document(metadata={'producer': 'Skia/PDF m118 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'data\\pdf\\170 Python projects.pdf', 'file_path': 'data\\pdf\\170 Python projects.pdf', 'total_pages': 15, 'format': 'PDF 1.4', 'title': '170 Python Projects', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page':

In [ ]:
# Install/upgrade langchain-text-splitters from within the notebook kernel
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'langchain-text-splitters'])
print('langchain-text-splitters installation complete')


  Attempting uninstall: langsmith

    Found existing installation: langsmith 0.3.42

    Uninstalling langsmith-0.3.42:

      Successfully uninstalled langsmith-0.3.42

   ---------------------------------------- 0/3 [langsmith]
   ---------------------------------------- 0/3 [langsmith]
  Attempting uninstall: langchain-core
   ---------------------------------------- 0/3 [langsmith]
    Found existing installation: langchain-core 0.3.59
   ---------------------------------------- 0/3 [langsmith]
   ------------- -------------------------- 1/3 [langchain-core]
    Uninstalling langchain-core-0.3.59:
   ------------- -------------------------- 1/3 [langchain-core]
      Successfully uninstalled langchain-core-0.3.59
   ------------- -------------------------- 1/3 [langchain-core]
   ------------- -------------------------- 1/3 [langchain-core]
   ------------- -------------------------- 1/3 [langchain-core]
   ------------- -------------------------- 1/3 [langchain-core]
   --------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.25 requires langsmith<0.4,>=0.1.17, but you have langsmith 0.4.37 which is incompatible.
langchain-community 0.3.23 requires langsmith<0.4,>=0.1.125, but you have langsmith 0.4.37 which is incompatible.


In [33]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [10]:
# --- Document Chunking ---
# Import the text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 1. Initialize the Text Splitter
# Set the maximum size of each chunk and the overlap between chunks.
# - chunk_size: The maximum number of characters (or tokens, depending on the length function) per chunk. 
#   A common starting point is 500-1000 characters.
# - chunk_overlap: The number of characters to overlap between adjacent chunks. This helps maintain context. 
#   A common starting point is 10-20% of chunk_size (e.g., 50-100).
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100 
)

# 2. Split the documents
# The split_documents method takes the list of loaded documents and returns a new, split list.
chunked_documents = text_splitter.split_documents(pdf_documents)

# 3. View the results
print(f"Documents split into {len(chunked_documents)} chunks.")
print("\nExample of the first chunk:")
print(chunked_documents[0].page_content) 
print(f"Metadata for the first chunk: {chunked_documents[0].metadata}")

Documents split into 29 chunks.

Example of the first chunk:
170 Python
Projects
with Source Code & Resources
A collection of simple Python mini projects to enhance your Python skills.
Himanshu Ramchandani
https://www.linkedin.com/in/hemansnation/
Metadata for the first chunk: {'producer': 'Skia/PDF m118 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'data\\pdf\\170 Python projects.pdf', 'file_path': 'data\\pdf\\170 Python projects.pdf', 'total_pages': 15, 'format': 'PDF 1.4', 'title': '170 Python Projects', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}


In [20]:
### Converting the text to embeddings
#texts=[doc.page_content for doc in chunks]
#texts

### Embedding and VectorStoreDB

In [25]:
pip uninstall protobuf -y


Found existing installation: protobuf 5.29.3
Uninstalling protobuf-5.29.3:
  Successfully uninstalled protobuf-5.29.3
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [26]:
pip install protobuf==3.20.3

   ---------------------------------------- 0.0/904.2 kB ? eta -:--:--
   ---------------------------------------- 904.2/904.2 kB 14.0 MB/s  0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.70.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.


In [27]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

RuntimeError: Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_utils because of the following error (look up to see its traceback):
Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

In [22]:
import sys, subprocess, importlib.util
print("Kernel Python:", sys.executable)
spec = importlib.util.find_spec("chromadb")
if spec is None:
    print("chromadb not found in this kernel. Installing...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'chromadb'])
    print("\nInstalled chromadb into kernel Python. Please restart the kernel and re-run the import cell.")
else:
    print("chromadb already available in this kernel.")

Kernel Python: c:\Users\niazs\anaconda3\python.exe
chromadb already available in this kernel.


In [23]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


RuntimeError: Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_utils because of the following error (look up to see its traceback):
Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

In [17]:
class EmbeddingManger:
    """Handle document embedding generation using SentenceTransformer. 
    we have to run this into the powershell:huggingface-cli logout. This will clear any cached credentials that might be causing the issue."""
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):                   
        """ initialize the embedding manager
        Args:
            model_name (str): HuggingFace model name for sentence embeddings.
        """
        self.model_name = model_name ##SentenceTransformer(model_name)
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.
        Args:
            texts (List[str]): List of text strings to embed.
        Returns:
            np.ndarray: Array of embeddings with shape (len(texts), embedding_dim).
        """
        if not self.model:
            raise ValueError("Model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]}")
        return embeddings


## initialize the embedding manager

embedding_manager = EmbeddingManger()
embedding_manager

NameError: name 'List' is not defined

### VectorStore

In [27]:
import os
import uuid
from typing import List, Any
import chromadb
# Assuming EmbeddingManger and Document classes are defined elsewhere or passed correctly

# --- Mock classes for testing purposes (you must replace these with your actual classes) ---
class EmbeddingManger:
    def generate_embeddings(self, texts: List[str]):
        # Mock method for the logic below: this should generate the embeddings
        pass 
    def __len__(self):
        # Mock method to allow len() call, though the logic needs to change
        return 0

class Document:
    def __init__(self, page_content, metadata):
        self.page_content = page_content
        self.metadata = metadata
# ----------------------------------------------------------------------------------------


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store."""

    def __init__(self, collection_name: str="pdf_documents", persist_directory: str = "data/vector_store_chroma"):
        """Initialize the vector store."""
        self.collection_name = collection_name
        # FIX 1: Removed space typo from persist_directory
        self.persist_directory = persist_directory.strip() 
        self.client = None
        self.collection = None  
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            # Create or get collection
            # NOTE: If you are using an external embedding model (like all-MiniLM-L6-v2), 
            # you will need to pass an embedding function here: embedding_function=your_ef
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings from RAG project"}
            )
            print(f"Vector Store '{self.collection_name}' initialized in '{self.persist_directory}'.")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Document], embedding_manager: EmbeddingManger):
        """Add documents and their embeddings to the vector store."""
        
        # FIX 2: Generate embeddings FIRST, then check length against documents
        texts_to_embed = [doc.page_content for doc in documents]
        embeddings = embedding_manager.generate_embeddings(texts_to_embed) 
        
        if len(documents) != len(embeddings):
             # This should ideally not happen if generate_embeddings works correctly
             raise ValueError("Number of documents and generated embeddings must match.") 
             
        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for chromadb
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = [] 

        # FIX 3: Zip documents with the generated embeddings list
        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            # FIX 4: Corrected f-string syntax for unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" 
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Append document text
            documents_texts.append(doc.page_content)
            
            # FIX 5: Use 'emb' (the variable from zip) and ensure it's converted if necessary
            # Assuming 'emb' is a numpy array slice, we convert it to a list for ChromaDB
            embeddings_list.append(emb.tolist()) 

        # Add to chromadb collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

# Now you can initialize the store (assuming imports are correct)
vector_store = VectorStore()
vector_store

Vector Store 'pdf_documents' initialized in 'data/vector_store_chroma'.
Existing documents in collection: 0


In [16]:
# 1. Create the list of texts
chunks = text_splitter.split_documents(pdf_documents)
texts = [doc.page_content for doc in chunks]

# 2. Convert the list of texts into embeddings (The missing crucial step)
# This is where you would call your EmbeddingManger:
embeddings = embedding_manager.generate_embeddings(texts) 

# embeddings will now be a NumPy array ready for your VectorStore

NameError: name 'embedding_manager' is not defined